# Extraire les entités de la presse de 1914

Quelles personnes, quels lieux, quelles organisations un passage nomme-t-il ? Trois méthodes, du plus rapide au plus souple, comparées sur 40 passages annotés à la main :

1. spaCy, ses étiquettes fixes
2. GLiNER, des étiquettes au choix
3. Un LLM qui remplit un schéma
4. Relier les noms à une liste de référence
5. Comparer, et repérer les inventions

Ce notebook a besoin d'un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données et outils

Le LLM tourne dans [Ollama](https://ollama.com), sur la machine Colab : la cellule suivante l'installe et télécharge le modèle en arrière-plan, pendant que vous travaillez sur spaCy et GLiNER.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/gold.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q gliner rapidfuzz
!python -m spacy download fr_core_news_md > /dev/null

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import json

import pandas as pd
from rapidfuzz import fuzz

gold = pd.read_json("gold.jsonl", lines=True)
FIELDS = ["people", "places", "organizations"]
print(len(gold), "passages annotés")
gold.head(3)

`evaluate(predictions)` compare des extractions (une liste de dictionnaires `{"people": [...], "places": [...], "organizations": [...]}`, dans l'ordre de `gold`) aux annotations : pour chaque champ, la précision (la part des entités extraites qui sont justes), le rappel (la part des entités attendues qui sont trouvées) et leur F1. La correspondance est souple : deux noms correspondent si leur similarité de Levenshtein (`fuzz.ratio`) dépasse 85 sur 100, ou si l'un contient l'autre (« Jaurès » et « Jean Jaurès »).

In [ ]:
def normalize(name: str) -> str:
    return " ".join(name.lower().replace("’", "'").split())


def same(a: str, b: str) -> bool:
    a, b = normalize(a), normalize(b)
    return fuzz.ratio(a, b) >= 85 or (min(len(a), len(b)) >= 4 and (a in b or b in a))


def evaluate(predictions: list[dict[str, list[str]]]) -> pd.DataFrame:
    rows = {}
    for field in FIELDS:
        found = expected = right_found = right_expected = 0
        for prediction, (_, row) in zip(predictions, gold.iterrows()):
            predicted = list(dict.fromkeys(prediction.get(field, [])))
            truth = row[field]
            found += len(predicted)
            expected += len(truth)
            right_found += sum(any(same(p, t) for t in truth) for p in predicted)
            right_expected += sum(any(same(p, t) for p in predicted) for t in truth)
        precision = right_found / max(found, 1)
        recall = right_expected / max(expected, 1)
        rows[field] = {"précision": precision, "rappel": recall, "F1": 2 * precision * recall / max(precision + recall, 1e-9)}
    return pd.DataFrame(rows).T.round(2)


results = {}

## 1. spaCy

Chargez `fr_core_news_md` et extrayez les entités de chaque passage : `PER` pour les personnes, `LOC` pour les lieux, `ORG` pour les organisations. Évaluez, et rangez le tableau dans `results["spaCy"]`. Combien de temps pour les 40 passages ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
import time

import spacy

nlp = spacy.load("fr_core_news_md")
LABELS = {"PER": "people", "LOC": "places", "ORG": "organizations"}
start = time.time()
spacy_predictions = []
for doc in nlp.pipe(gold.text):
    prediction = {field: [] for field in FIELDS}
    for ent in doc.ents:
        if ent.label_ in LABELS:
            prediction[LABELS[ent.label_]].append(ent.text)
    spacy_predictions.append(prediction)
print(f"{time.time() - start:.1f} s")
results["spaCy"] = evaluate(spacy_predictions)
results["spaCy"]

## 2. GLiNER

[GLiNER](https://github.com/urchade/GLiNER) trouve les entités des étiquettes qu'on lui donne, sans entraînement. Chargez `urchade/gliner_multi-v2.1` (sur le GPU, `.to("cuda")`) et extrayez les entités des étiquettes `["personne", "lieu", "organisation"]` (`model.predict_entities(texte, étiquettes, threshold=0.4)`). Évaluez. Essayez d'autres formulations des étiquettes (« journal », « parti politique »…) ou un autre seuil.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from gliner import GLiNER

gliner = GLiNER.from_pretrained("urchade/gliner_multi-v2.1").to("cuda")
FIELD_OF = {"personne": "people", "lieu": "places", "organisation": "organizations"}
start = time.time()
gliner_predictions = []
for text in gold.text:
    prediction = {field: [] for field in FIELDS}
    for entity in gliner.predict_entities(text, list(FIELD_OF), threshold=0.4):
        prediction[FIELD_OF[entity["label"]]].append(entity["text"])
    gliner_predictions.append(prediction)
print(f"{time.time() - start:.1f} s")
results["GLiNER"] = evaluate(gliner_predictions)
results["GLiNER"]

## 3. Un LLM qui remplit un schéma

Le modèle tourne dans Ollama, qui offre une API compatible OpenAI : le même code fonctionnerait avec une API commerciale, en changeant l'adresse et la clé.

Décrivez le résultat attendu avec une classe Pydantic `Entities` (trois listes de chaînes), puis écrivez `extract(text)` qui demande au modèle les entités du passage, avec `client.chat.completions.parse(model=MODEL, messages=..., response_format=Entities, temperature=0)` : la réponse respecte forcément le schéma. Votre prompt doit donner les consignes d'annotation (ci-dessous). Lancez l'extraction des 40 passages **en parallèle** (`ThreadPoolExecutor(8)`), mesurez le temps, évaluez.

Consignes d'annotation : chaque entité nommée une fois, telle qu'écrite dans le texte (erreurs d'OCR comprises) ; les personnes sans leur titre (M., Mme, sir) ; les lieux (pays, villes, rues, bâtiments) ; les organisations nommées (assemblées, partis, syndicats, journaux, agences) ; pas de mentions génériques (« le gouvernement »).

In [ ]:
pull.wait()  # le modèle est téléchargé

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# Un premier appel charge le modèle sur le GPU, où il reste : avant les calculs en parallèle.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1)
print(MODEL, "prêt")

In [ ]:
# Votre code ici

### Solution

In [ ]:
from concurrent.futures import ThreadPoolExecutor


class Entities(BaseModel):
    people: list[str]
    places: list[str]
    organizations: list[str]


INSTRUCTIONS = """Extrais les entités nommées de ce passage de presse de 1914.
- people : les personnes, sans leur titre (M., Mme, sir, lord) ;
- places : les lieux (pays, régions, villes, rues, bâtiments) ;
- organizations : les organisations nommées (assemblées, partis, syndicats, journaux, agences).
Chaque entité une fois, telle qu'elle est écrite dans le texte, erreurs comprises.
Pas de mentions génériques comme « le gouvernement ». N'invente rien."""


def extract(text: str) -> dict[str, list[str]]:
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": text}],
        response_format=Entities,
        temperature=0,
    )
    return response.choices[0].message.parsed.model_dump()


start = time.time()
with ThreadPoolExecutor(8) as pool:
    llm_predictions = list(pool.map(extract, gold.text))
print(f"{time.time() - start:.1f} s")
results["LLM"] = evaluate(llm_predictions)
results["LLM"]

## 4. Relier les noms à une liste de référence

Les mêmes personnes apparaissent sous des formes différentes (« Jaurès », « Jean Jaurès », « M. Jaurès »), parfois mal reconnues par l'OCR. Pour chaque personne extraite par le LLM, cherchez dans `REFERENCE` le nom le plus proche (`rapidfuzz.process.extractOne(nom, REFERENCE, scorer=fuzz.WRatio)`), et gardez-le si le score dépasse 85. Quelles personnes du corpus sont reliées ?

In [ ]:
REFERENCE = [
    "Jean Jaurès", "Joseph Caillaux", "Henriette Caillaux", "Gaston Calmette", "Raymond Poincaré",
    "René Viviani", "Georges Clemenceau", "Jules Guesde", "Léon Jouhaux", "Édouard Vaillant",
    "Edward Grey", "Herbert Asquith", "Paul Cambon", "Raoul Villain", "Gabriel Hanotaux",
    "Antonino di San Giuliano", "Émile Vandervelde", "John Simon", "Woodrow Wilson",
]

In [ ]:
# Votre code ici

### Solution

In [ ]:
from rapidfuzz import process

for name in sorted({name for prediction in llm_predictions for name in prediction["people"]}):
    match, score, _ = process.extractOne(name, REFERENCE, scorer=fuzz.WRatio)
    if score >= 85:
        print(f"{name:25} -> {match} ({score:.0f})")

## 5. Comparer, et repérer les inventions

Rassemblez les F1 des trois méthodes dans un tableau. Puis cherchez les **inventions** du LLM : les entités extraites qui n'apparaissent pas dans le texte du passage (`fuzz.partial_ratio(normalize(entité), normalize(texte)) < 85`). Y en a-t-il ? Laquelle des trois méthodes choisiriez-vous pour 10 millions de passages ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
print(pd.DataFrame({name: table["F1"] for name, table in results.items()}))

for prediction, text in zip(llm_predictions, gold.text):
    for field in FIELDS:
        for entity in prediction[field]:
            if fuzz.partial_ratio(normalize(entity), normalize(text)) < 85:
                print(f"{field:13} {entity!r} n'est pas dans le texte")

# spaCy est de loin le plus rapide, mais ses étiquettes sont fixes et il se trompe sur un
# texte bruité ; GLiNER est un bon compromis ; le LLM est le plus souple et en général le
# plus juste, mais le plus lent et le plus cher : à 10 millions de passages, on l'utiliserait
# pour étiqueter des exemples, puis on affinerait un petit modèle (TP suivant).